# Yolo V8 Custom Data set

----

Conda env : [cv_playgrounds](../README.md#setup-a-conda-environment)

----

- Ref: 
    - https://learnopencv.com/train-yolov8-on-custom-dataset/

In [1]:
!nvidia-smi

Thu Nov 20 18:45:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 29%   41C    P5             30W /  250W |     441MiB /  11264MiB |     20%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download custom data for pothole
- Ref: https://public.roboflow.com/object-detection/pothole

In [2]:
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)
Path("./temp_model").mkdir(exist_ok=True, parents=True)

In [2]:
! wget https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1 -O ./temp_data/pothole_dataset_v8.zip


--2025-11-20 17:01:06--  https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.13.18, 2620:100:6057:18::a27d:d12
Connecting to www.dropbox.com (www.dropbox.com)|162.125.13.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/em7irx9n0ukb2g8jw9kx0/pothole_dataset_v8.zip?rlkey=launc4guu0wvvib144y8mt4n7&dl=1 [following]
--2025-11-20 17:01:06--  https://www.dropbox.com/scl/fi/em7irx9n0ukb2g8jw9kx0/pothole_dataset_v8.zip?rlkey=launc4guu0wvvib144y8mt4n7&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucaa07e57acd8186bed00cd8cdf2.dl.dropboxusercontent.com/cd/0/inline/C1iBXJAt0ETXO1f5v0aoBiEY9P0S43EvgNNh7EtMggpgkNLimERmTRQQKZLzcVoa7b78w46kQjg_OcjJpIhAZdEBhA95zDvh_cqlqPi9IZuSP5U9lNY5PukiV1_ELceIjjmxBSYkKSR5Ajuw1w8SffTe/file?dl=1# [following]
--2025-11-20 17:01:07--  https://ucaa07e57acd8186

In [3]:
import zipfile

zipfile_path = "./temp_data/pothole_dataset_v8.zip"
dataset_path = "./temp_data/pothole_dataset_v8"
with zipfile.ZipFile(zipfile_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

### Generate a test data

In [11]:
!ffmpeg -framerate 30 -pattern_type glob -i 'temp_data/pothole_dataset_v8/pothole_dataset_v8/valid/images/G*.jpg' -c:v libx264 -pix_fmt yuv420p temp_data/test_video.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Fine Tune Yolo8-nano

In [2]:
from ultralytics import YOLO
 
# Load the model.
model = YOLO('./temp_model/yolov8n.pt')
 
# Training.
results = model.train(
   data='pothole_yolov8.yaml',
   imgsz=1280,
   epochs=50,
   batch=8,
   project = "temp_output_yolov8n_v8_50e",
   name='yolov8n_v8_50e'
)

New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pothole_yolov8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./temp_model/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_v8_50e

In [2]:
!yolo task=detect mode=val model=temp_output_yolov8n_v8_50e/yolov8n_v8_50e/weights/best.pt name=yolov8n_v8_50e_eval data=pothole_yolov8.yaml imgsz=1280


Ultralytics 8.3.230 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 279.1±108.4 MB/s, size: 433.0 KB)
val: Scanning /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/pothole_dataset_v8/pothole_dataset_v8/valid/labels.cache... 271 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 271/271 7.2Mit/s 0.0ss
val: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/pothole_dataset_v8/pothole_dataset_v8/valid/images/G0011603.jpg: 1 duplicate labels removed
val: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/pothole_dataset_v8/pothole_dataset_v8/valid/images/G0011614.jpg: 1 duplicate labels removed
val: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/pothole_dataset_v8/pothole_dataset_v8/valid/images/G001

In [12]:
!yolo task=detect mode=predict model=temp_output_yolov8n_v8_50e/yolov8n_v8_50e/weights/best.pt source=temp_data/test_video.mp4 show=True imgsz=1280 name=yolov8n_v8_50e_infer1280 hide_labels=True

WARNING ⚠️ 'hide_labels' is deprecated and will be removed in the future. Use 'show_labels' instead.
Ultralytics 8.3.230 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

video 1/1 (frame 1/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 (no detections), 32.5ms
video 1/1 (frame 2/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 1 pothole, 7.3ms
video 1/1 (frame 3/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 1 pothole, 12.9ms
video 1/1 (frame 4/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 3 potholes, 11.9ms
video 1/1 (frame 5/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_vide

### Fine Tune Yolo8-small

In [3]:
from ultralytics import YOLO
 
# Load the model.
model = YOLO('./temp_model/yolov8s.pt')
 
# Training.
results = model.train(
   data='pothole_yolov8.yaml',
   imgsz=1280,
   epochs=50,
   batch=8,
   project = "temp_output_yolov8s_v8_50e",
   name='yolov8s_v8_50e'
)

New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pothole_yolov8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./temp_model/yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_v8_50e

In [13]:
!yolo task=detect mode=predict model=temp_output_yolov8s_v8_50e/yolov8s_v8_50e/weights/best.pt source=temp_data/test_video.mp4 show=True imgsz=1280 name=yolov8s_v8_50e_infer1280 hide_labels=True

WARNING ⚠️ 'hide_labels' is deprecated and will be removed in the future. Use 'show_labels' instead.
Ultralytics 8.3.230 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

video 1/1 (frame 1/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 1 pothole, 41.8ms
video 1/1 (frame 2/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 2 potholes, 10.4ms
video 1/1 (frame 3/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 1 pothole, 17.2ms
video 1/1 (frame 4/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 2 potholes, 16.8ms
video 1/1 (frame 5/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.

### Fine Tune Yolo8-medium

In [1]:
from ultralytics import YOLO
 
# Load the model.
model = YOLO('./temp_model/yolov8m.pt')
 
# Training.
results = model.train(
   data='pothole_yolov8.yaml',
   imgsz=1280,
   epochs=50,
   batch=4,
   project = "temp_output_yolov8m_v8_50e",
   name='yolov8m_v8_50e'
)

New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pothole_yolov8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./temp_model/yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_v8_50e

In [14]:
!yolo task=detect mode=predict model=temp_output_yolov8m_v8_50e/yolov8m_v8_50e/weights/best.pt source=temp_data/test_video.mp4 show=True imgsz=1280 name=yolov8m_v8_50e_infer1280 hide_labels=True

WARNING ⚠️ 'hide_labels' is deprecated and will be removed in the future. Use 'show_labels' instead.
Ultralytics 8.3.230 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10815MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

video 1/1 (frame 1/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 (no detections), 33.5ms
video 1/1 (frame 2/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 1 pothole, 25.8ms
video 1/1 (frame 3/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 (no detections), 25.6ms
video 1/1 (frame 4/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/test_video.mp4: 736x1280 3 potholes, 24.8ms
video 1/1 (frame 5/186) /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/2DCV/Yolo/temp_data/